# Week 4: Plotting global launch sites (Capstone 1 week)

**Track:** Ground Station Operator (Beginner)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/4/](https://launchdetect.com/academy/week/4/)  
**Track index:** [https://launchdetect.com/academy/ground-station-operator/](https://launchdetect.com/academy/ground-station-operator/)

---

_Track 1 culminates here: every active orbital launch pad on Earth, geocoded, attributed (country, operator, status, vehicles), styled, and mapped. The week 4 lab IS the capstone start — you finish it for cert 1._


## Why this week matters

Week 4 is the capstone of Track 1. Everything you learned in weeks 1-3 — coordinate systems, vector data, QGIS styling — comes together to produce a single deliverable: a citation-grade global atlas of every active orbital launch pad on Earth.

**Why does a citation-grade atlas matter?** Because data quality is the precondition for every analysis that follows. The atlas you build this week is the source-of-truth input to Weeks 9 (ground-station coverage), 10 (spaceport-to-orbit matching), and every subsequent geospatial query that involves a launch site. If your pad coordinates are off by 5 km, every downstream answer is off by at least 5 km — and most consumers of your atlas will trust your numbers without double-checking. **The atlas must be right.**

This is also the first time you'll exercise the rigor that distinguishes academic-grade from amateur work: every coordinate cited to a primary source, every attribute defensible, every styling decision intentional. The capstone grader expects this.


## Learning objectives

By the end of this lab you will be able to:

- Compile the world's active orbital launch pads into a GeoJSON
- Compute nearest-neighbor distances between spaceports
- Identify which spaceports share an inclination band
- Produce a styled global atlas map ready for publication


## Setup — and why these dependencies

Same `leafmap` + `pyproj` from earlier weeks. No new dependencies — this is a synthesis week.


In [ ]:
# Install required packages
!pip install -q leafmap[common] pyproj geopandas shapely


## The approach (and why)

Build the full atlas in two passes:

**Pass 1 (this notebook):** Scaffold the data structure. Start with the Week 3 subset of 16 pads. Add the metadata fields the capstone rubric requires: `first_orbital_launch`, `latest_orbital_launch`, `vehicles`, `status`, `source`. Write your starter GeoJSON to disk and visualize.

**Pass 2 (your work, off-notebook):** Extend to all currently-active orbital pads worldwide (the capstone target is 18+; the real number is closer to 25). For each, cite the primary source (LaunchDetect atlas, UN COPUOS registry, operator press kit, FAA AST license). Style in QGIS, compose a print layout, export an A2 PDF.

**Why two passes?** Because the second pass is the actual capstone work — it takes 8-15 hours of research, not 1 hour of coding. Don't try to do it all in this notebook; the notebook gets you started, and the deliverable is the polished output.


In [ ]:
# Week 4: Global Launch Site Atlas — CAPSTONE 1 START
import leafmap.foliumap as leafmap
import json

# Extend the Week 3 pad list to the full active-orbital roster (target: 20+).
# Cite primary sources in the "source" field for each feature.
pads = []  # TODO: populate from launchdetect.com/atlas/ + operator press kits

features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
        "properties": {k: v for k, v in p.items() if k not in ("lat", "lon")}
    }
    for p in pads
]
fc = {"type": "FeatureCollection", "features": features}

with open("launch-sites.geojson", "w") as f:
    json.dump(fc, f, indent=2)
print(f"Wrote launch-sites.geojson — {len(features)} features")

m = leafmap.Map(center=[20, 0], zoom=2)
if features:
    m.add_geojson(fc, style={"color": "#ff6b35"})
m

# TODO: extend `pads` to all active orbital sites with cited coordinates,
# style by operator in QGIS, export an A2 PDF. Deliverable: launch-sites.geojson + atlas.pdf.


## What just happened — and why it works

What you have right now is the *skeleton* of a citation-grade GeoJSON. Each feature follows the GeoJSON spec exactly: `type: 'Feature'`, with `geometry` (a Point with [lon, lat] coordinates) and `properties` (the attributes). A FeatureCollection wraps the features in a single envelope so it parses as one JSON document.

This structure is what every GIS tool can ingest natively: QGIS opens it, Mapbox/MapLibre serves it as vector tiles, PostGIS imports it via `ST_GeomFromGeoJSON`. **Pick GeoJSON for vector data exchange whenever you have the choice** — it's open, human-readable, and lossless.


## Common gotchas

- **'Active' is a judgment call.** A pad with a flight in the last 24 months is definitely active. Six months past last flight? Probably active. Two years? Probably not. Pick a threshold and document it.
- **First-orbital-launch is the date of the first orbital ATTEMPT from the pad, not necessarily a success.** Cite carefully — the original Atlas-Agena pad that became LC-39A had decades of variant naming before it became 'LC-39A'.
- **Don't conflate spaceport, complex, and pad.** Kennedy Space Center is a spaceport. LC-39 is a complex. LC-39A is a pad. Decide which granularity your atlas represents and stay consistent.


## Doing this in QGIS (alternative path)

**Final QGIS print-layout checklist (capstone-grade):**

- **Page size A2 (594 × 420 mm).** Standard poster size; readable at 1 m viewing.
- **Map fills ~70% of the page.** Leave room for legend, title, scale bar, attribution.
- **CRS for the map item: Robinson (ESRI:54030).** Best for global thematic maps — balances area and shape distortion better than Mercator. *Why not Mercator:* polar distortion makes Arctic launch sites look bigger than equatorial ones.
- **Title: 'Active Orbital Launch Pads, 2026'.** Date is essential — this dataset snapshots a moment.
- **Legend grouped by operator** (categorized symbology from Week 3 transfers directly).
- **Scale bar in km** — meters too small to read at global zoom, miles inappropriate for an international audience.
- **North arrow** — implicit at this scale but conventionally required.
- **Attribution: 'Sources: LaunchDetect.com/atlas/, UN COPUOS registry, operator press kits as of YYYY-MM-DD'.** Citation density is what separates publication-quality from amateur.
- **Export: PDF, 300 dpi, 'rasterize' option off** (so vector text stays crisp at any zoom).

Save the `.qgz` project alongside the PDF and the source GeoJSON — that triad is the reproducible-publication package.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/4/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
